In [10]:
import pandas as pd
from scipy.stats import mannwhitneyu, ranksums
from statsmodels.stats.multitest import multipletests
import numpy as np
seeds = [42,0,1,2,3]
for seed in seeds:
    datasets = [f'avatarsk5_{seed}', f'avatarsk10_{seed}',f'ctgan_{seed}',
           f'gaussiancopula_{seed}', f'synthpop_{seed}', f'tvae_{seed}']
    for dataset in datasets:
        print(f'---{dataset}---')
        signature_scores = pd.read_csv(f'Jerby_ImmuneCell/{dataset}.csv', index_col = 0)
        # if dataset == 'Origin':
        #     metadata = 'original_data'
        original_data = pd.read_csv(f'../../Data/{dataset}.csv')

        #Add labels
        responderCrit = original_data['BR'].isin(['CR','PR'])
        progressorCrit = original_data['BR'].isin(['PD'])
        
        responders = original_data[responderCrit].index
        progressors = original_data[progressorCrit].index
        
        original_data.loc[responders,"Labels"] = 'Responder'
        original_data.loc[progressors,"Labels"] = 'Progressor'

        #Overall
        responders = original_data[original_data['Labels']=='Responder']['Patient_ID'].values.tolist()
        progressors = original_data[original_data['Labels']=='Progressor']['Patient_ID'].values.tolist()
        cell_types = signature_scores.columns.tolist()
        pvalues = []
        diff_means = []
        for cell_type in cell_types: 
            reponders_score = signature_scores.loc[responders,cell_type].values.tolist()
            progressors_score = signature_scores.loc[progressors,cell_type].values.tolist()
            diff_mean = np.mean(reponders_score) - np.mean(progressors_score)
            diff_means.append(diff_mean)
            _,pvalue = ranksums(reponders_score, progressors_score)
            pvalues.append(pvalue)
        _, pvals_corrected, _, _ = multipletests(pvalues, alpha=0.05, method='fdr_bh')

        #IpiTreated
        ipi_treated_df = original_data[original_data['daysBiopsyAfterIpiStart']=='postIpi']
        responders_ipi = ipi_treated_df[ipi_treated_df['Labels']=='Responder']['Patient_ID'].values.tolist()
        progressors_ipi = ipi_treated_df[ipi_treated_df['Labels']=='Progressor']['Patient_ID'].values.tolist()
        
        pvalues_ipi = []
        for cell_type in cell_types: 
            reponders_score = signature_scores.loc[responders_ipi,cell_type].values.tolist()
            progressors_score = signature_scores.loc[progressors_ipi,cell_type].values.tolist()
            _,pvalue_ipi = ranksums(reponders_score, progressors_score)
            pvalues_ipi.append(pvalue_ipi)
        _, pvals_corrected_ipi, _, _ = multipletests(pvalues_ipi, alpha=0.05, method='fdr_bh')

        #Ipi Naive
        ipinaive_treated_df = original_data[original_data['daysBiopsyAfterIpiStart']=='noIpi']
        responders_naiveipi = ipinaive_treated_df[ipinaive_treated_df['Labels']=='Responder']['Patient_ID'].values.tolist()
        progressors_naiveipi = ipinaive_treated_df[ipinaive_treated_df['Labels']=='Progressor']['Patient_ID'].values.tolist()
        
        pvalues_ipinaive = []
        for cell_type in cell_types: 
            reponders_score = signature_scores.loc[responders_naiveipi,cell_type].values.tolist()
            progressors_score = signature_scores.loc[progressors_naiveipi,cell_type].values.tolist()
            _,pvalue_ipinaive = ranksums(reponders_score, progressors_score)
            pvalues_ipinaive.append(pvalue_ipinaive)
        _, pvals_corrected_ipinaive, _, _ = multipletests(pvalues_ipinaive, alpha=0.05, method='fdr_bh')

        result_dict = {
                'Cell Type': cell_types,
                'P_values': pvalues,
                'Q_values': pvals_corrected,
                'Diff_Mean': diff_means,
                'P_values_IpiTreated': pvalues_ipi,
                'Q_values_IpiTreated': pvals_corrected_ipi,
                'P_values_IpiNaive': pvalues_ipinaive,
                'Q_values_IpiNaive': pvals_corrected_ipinaive,
            }
        df_res = pd.DataFrame(result_dict)
        df_res.to_csv(f'ResultsDA_Jerby/Seed_{seed}/{dataset}.csv')

---avatarsk5_42---
---avatarsk10_42---
---ctgan_42---
---gaussiancopula_42---
---synthpop_42---
---tvae_42---
---avatarsk5_0---
---avatarsk10_0---
---ctgan_0---
---gaussiancopula_0---
---synthpop_0---
---tvae_0---
---avatarsk5_1---
---avatarsk10_1---
---ctgan_1---
---gaussiancopula_1---
---synthpop_1---
---tvae_1---
---avatarsk5_2---
---avatarsk10_2---
---ctgan_2---
---gaussiancopula_2---
---synthpop_2---
---tvae_2---
---avatarsk5_3---
---avatarsk10_3---
---ctgan_3---
---gaussiancopula_3---
---synthpop_3---
---tvae_3---


In [15]:
import pandas as pd
dataset = 'original_data'
original_data = pd.read_csv(f'../../Data/{dataset}.csv')

#Add labels
responderCrit = original_data['BR'].isin(['CR','PR'])
progressorCrit = original_data['BR'].isin(['PD'])

responders = original_data[responderCrit].index
progressors = original_data[progressorCrit].index

original_data.loc[responders,"Labels"] = 'Responder'
original_data.loc[progressors,"Labels"] = 'Progressor'

print('Responder/Progressor', original_data['Labels'].value_counts())
print('Pencentage', len(responders)/len(progressors))

Responder/Progressor Labels
Progressor    56
Responder     47
Name: count, dtype: int64
Pencentage 0.8392857142857143


In [18]:
seeds = [0,1,2,3,42]
for seed in seeds: 
    print(seed)
    dataset = f'tvae_{seed}'
    original_data = pd.read_csv(f'../../Data/{dataset}.csv')
    
    #Add labels
    responderCrit = original_data['BR'].isin(['CR','PR'])
    progressorCrit = original_data['BR'].isin(['PD'])
    
    responders = original_data[responderCrit].index
    progressors = original_data[progressorCrit].index
    
    original_data.loc[responders,"Labels"] = 'Responder'
    original_data.loc[progressors,"Labels"] = 'Progressor'
    
    print('Responder/Progressor', original_data['Labels'].value_counts())
    print('Pencentage', len(responders)/len(progressors))

0
Responder/Progressor Labels
Progressor    75
Responder     28
Name: count, dtype: int64
Pencentage 0.37333333333333335
1
Responder/Progressor Labels
Responder     97
Progressor    14
Name: count, dtype: int64
Pencentage 6.928571428571429
2
Responder/Progressor Labels
Responder     85
Progressor    23
Name: count, dtype: int64
Pencentage 3.6956521739130435
3
Responder/Progressor Labels
Progressor    66
Responder     52
Name: count, dtype: int64
Pencentage 0.7878787878787878
42
Responder/Progressor Labels
Progressor    109
Responder      11
Name: count, dtype: int64
Pencentage 0.10091743119266056


In [9]:

signature_scores = pd.read_csv(f'Jerby_ImmuneCell/Origin.csv', index_col = 0)

original_data = pd.read_csv(f'../../Data/original_data.csv')

#Add labels
responderCrit = original_data['BR'].isin(['CR','PR'])
progressorCrit = original_data['BR'].isin(['PD'])

responders = original_data[responderCrit].index
progressors = original_data[progressorCrit].index

original_data.loc[responders,"Labels"] = 'Responder'
original_data.loc[progressors,"Labels"] = 'Progressor'

#Overall
responders = original_data[original_data['Labels']=='Responder']['Patient'].values.tolist()
progressors = original_data[original_data['Labels']=='Progressor']['Patient'].values.tolist()
cell_types = signature_scores.columns.tolist()
pvalues = []
diff_means = []
for cell_type in cell_types: 
    reponders_score = signature_scores.loc[responders,cell_type].values.tolist()
    progressors_score = signature_scores.loc[progressors,cell_type].values.tolist()
    diff_mean = np.mean(reponders_score) - np.mean(progressors_score)
    diff_means.append(diff_mean)
    _,pvalue = ranksums(reponders_score, progressors_score)
    pvalues.append(pvalue)
_, pvals_corrected, _, _ = multipletests(pvalues, alpha=0.05, method='fdr_bh')

#IpiTreated
ipi_treated_df = original_data[original_data['daysBiopsyAfterIpiStart']=='postIpi']
responders_ipi = ipi_treated_df[ipi_treated_df['Labels']=='Responder']['Patient'].values.tolist()
progressors_ipi = ipi_treated_df[ipi_treated_df['Labels']=='Progressor']['Patient'].values.tolist()

pvalues_ipi = []
for cell_type in cell_types: 
    reponders_score = signature_scores.loc[responders_ipi,cell_type].values.tolist()
    progressors_score = signature_scores.loc[progressors_ipi,cell_type].values.tolist()
    _,pvalue_ipi = ranksums(reponders_score, progressors_score)
    pvalues_ipi.append(pvalue_ipi)
_, pvals_corrected_ipi, _, _ = multipletests(pvalues_ipi, alpha=0.05, method='fdr_bh')

#Ipi Naive
ipinaive_treated_df = original_data[original_data['daysBiopsyAfterIpiStart']=='noIpi']
responders_naiveipi = ipinaive_treated_df[ipinaive_treated_df['Labels']=='Responder']['Patient'].values.tolist()
progressors_naiveipi = ipinaive_treated_df[ipinaive_treated_df['Labels']=='Progressor']['Patient'].values.tolist()

pvalues_ipinaive = []
for cell_type in cell_types: 
    reponders_score = signature_scores.loc[responders_naiveipi,cell_type].values.tolist()
    progressors_score = signature_scores.loc[progressors_naiveipi,cell_type].values.tolist()
    _,pvalue_ipinaive = ranksums(reponders_score, progressors_score)
    pvalues_ipinaive.append(pvalue_ipinaive)
_, pvals_corrected_ipinaive, _, _ = multipletests(pvalues_ipinaive, alpha=0.05, method='fdr_bh')

result_dict = {
        'Cell Type': cell_types,
        'P_values': pvalues,
        'Q_values': pvals_corrected,
        'Diff_Mean': diff_means,
        'P_values_IpiTreated': pvalues_ipi,
        'Q_values_IpiTreated': pvals_corrected_ipi,
        'P_values_IpiNaive': pvalues_ipinaive,
        'Q_values_IpiNaive': pvals_corrected_ipinaive,
    }
df_res = pd.DataFrame(result_dict)
df_res.to_csv(f'ResultsDA_Jerby/Origin.csv')